# 01 — Data Audit

**NIFTY 50 Next-Day Direction Prediction**

This notebook documents all data quality findings for the three raw datasets:
- NIFTY 50 (nifty50.csv)
- Bank NIFTY (banknifty.csv)
- India VIX (indiavix.csv)

We also audit the pre-built `starter_features.csv` and identify critical issues including **leakage** and **multicollinearity**.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import sys
sys.path.insert(0, str(Path.cwd().parent) if 'notebooks' in str(Path.cwd()) else str(Path.cwd()))

DATA_DIR = Path('../data') if Path('../data').exists() else Path('data')
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 12

## 1. Load Raw Datasets

In [ ]:
nifty = pd.read_csv(DATA_DIR / 'nifty50.csv', parse_dates=['date']).sort_values('date').reset_index(drop=True)
bn = pd.read_csv(DATA_DIR / 'banknifty.csv', parse_dates=['date']).sort_values('date').reset_index(drop=True)
vix = pd.read_csv(DATA_DIR / 'indiavix.csv', parse_dates=['date']).sort_values('date').reset_index(drop=True)
starter = pd.read_csv(DATA_DIR / 'starter_features.csv', parse_dates=['date']).sort_values('date').reset_index(drop=True)

for name, df in [('NIFTY 50', nifty), ('Bank NIFTY', bn), ('India VIX', vix), ('Starter Features', starter)]:
    print(f'{name}: {df.shape[0]} rows, {df.shape[1]} cols, {df["date"].min().date()} to {df["date"].max().date()}')
    print(f'  Columns: {list(df.columns)}')
    print(f'  Nulls: {df.isnull().sum().sum()}')
    print()

## 2. Issue 1: VIX Missing 5 Dates

In [ ]:
nifty_dates = set(nifty['date'])
vix_dates = set(vix['date'])
bn_dates = set(bn['date'])

vix_missing = sorted(nifty_dates - vix_dates)
print(f'VIX missing {len(vix_missing)} dates present in NIFTY:')
for d in vix_missing:
    print(f'  {d.date()} ({d.strftime("%A")})')

## 3. Issue 2: Bank NIFTY Missing 1 Date

In [ ]:
bn_missing = sorted(nifty_dates - bn_dates)
print(f'Bank NIFTY missing {len(bn_missing)} date(s):')
for d in bn_missing:
    print(f'  {d.date()} ({d.strftime("%A")})')

## 4. Issue 3: NIFTY Zero-Volume Days (8)

In [ ]:
zero_vol = nifty[nifty['Volume'] == 0]
print(f'Zero-volume days: {len(zero_vol)}')
print(zero_vol[['date', 'Close', 'Volume']].to_string(index=False))

## 5. Issue 4: Saturday Trading Session

In [ ]:
saturdays = nifty[nifty['date'].dt.dayofweek == 5]
print(f'Saturday sessions in NIFTY: {len(saturdays)}')
print(saturdays[['date', 'Open', 'Close', 'Volume']].to_string(index=False))

## 6. Issue 5: Bank NIFTY Corporate Actions

In [ ]:
bn_adj_diff = bn[bn['Close'] != bn['Adj Close']]
print(f'Bank NIFTY rows where Adj Close != Close: {len(bn_adj_diff)} / {len(bn)}')

# NIFTY check
nifty_adj_diff = nifty[nifty['Close'] != nifty['Adj Close']]
print(f'NIFTY 50 rows where Adj Close != Close: {len(nifty_adj_diff)} / {len(nifty)}')
print('  → NIFTY is an index, no corporate actions. Safe to use Close directly.')

## 7. CRITICAL: ma5_smooth_signal Leakage Detection

In [ ]:
# Compute target
starter_with_target = starter.copy()
nifty_close = nifty.set_index('date')['Close']
starter_with_target = starter_with_target.merge(
    pd.DataFrame({'date': nifty['date'], 'close': nifty['Close'], 'next_ret': nifty['Close'].pct_change(1).shift(-1)}),
    on='date', how='inner'
)
starter_with_target['target'] = (starter_with_target['next_ret'] > 0).astype(int)

if 'ma5_smooth_signal' in starter_with_target.columns:
    corr_target = starter_with_target['ma5_smooth_signal'].corr(starter_with_target['target'])
    corr_next_ret = starter_with_target['ma5_smooth_signal'].corr(starter_with_target['next_ret'])
    nulls_head = starter_with_target['ma5_smooth_signal'].head(10).isnull().sum()
    nulls_tail = starter_with_target['ma5_smooth_signal'].tail(10).isnull().sum()
    
    print('ma5_smooth_signal Analysis:')
    print(f'  Correlation with target:     {corr_target:.4f}')
    print(f'  Correlation with next_ret:   {corr_next_ret:.4f}')
    print(f'  Nulls in first 10 rows:      {nulls_head}')
    print(f'  Nulls in last 10 rows:       {nulls_tail}')
    print()
    print('VERDICT: *** DROP ma5_smooth_signal — LOOK-AHEAD LEAKAGE ***')
    print('  Legitimate trailing MA would have nulls at BEGINNING, not END.')
else:
    print('ma5_smooth_signal not found in starter features')

## 8. Perfect Collinearity: ret_1d ↔ ret_zscore

In [ ]:
if 'ret_1d' in starter.columns and 'ret_zscore' in starter.columns:
    corr = starter['ret_1d'].corr(starter['ret_zscore'])
    print(f'Correlation ret_1d vs ret_zscore: {corr:.6f}')
    print('VERDICT: Perfect collinearity. Keep ret_1d (more interpretable), DROP ret_zscore.')
else:
    print('One or both features not found')

## 9. Summary of Actions Taken

| Issue | Action |
|-------|--------|
| VIX missing 5 dates | Forward-fill onto NIFTY date grid |
| Bank NIFTY missing 1 date | Forward-fill |
| 8 zero-volume days | Mark as NaN, handle in features |
| Saturday 2025-02-01 | Keep (real NSE special session) |
| Bank NIFTY Adj Close != Close | Use Adj Close for returns |
| ma5_smooth_signal | **DROPPED — look-ahead leakage** |
| ret_zscore | **DROPPED — perfect collinearity with ret_1d** |